# 2D Binary Pattern Matching — Comprehensive Testing Notebook
Tests three scenarios:
1. **Random Growth** — ablation over pattern & text sizes, binary random arrays
2. **Real-World QR Codes** — search for a Tetrimino (L-shape) in 100 QR code images
3. **Contrived Cases** — all-1s text with a hidden 3×3 block of 0s near the end

Primary metric: **equality-operation count**. Wall-clock time is recorded as a secondary metric.

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from PIL import Image

from algos.naive import naive_search
from algos.rabin_karp import rabin_karp_search, vectorized_rabin_karp
from algos.greedy import greedy_kmp_search, greedy_bm_search
from algos.bird import bird_bm, bird_kmp

ALGORITHMS = {
    'Naive':               naive_search,
    'Rabin-Karp':          rabin_karp_search,
    'Vectorized RK':       vectorized_rabin_karp,
    'Greedy KMP':          greedy_kmp_search,
    'Greedy BM':           greedy_bm_search,
    'Bird-KMP':            bird_kmp,
    'Bird-BM':             bird_bm,
}

print('All imports OK.')
print('Algorithms registered:', list(ALGORITHMS.keys()))

---
## Helper Utilities

In [ ]:
def make_donut(size: int) -> np.ndarray:
    """Square binary array of 0s wrapped by a border of 1s (donut shape)."""
    arr = np.zeros((size, size), dtype=np.int8)
    arr[0, :]  = 1
    arr[-1, :] = 1
    arr[:, 0]  = 1
    arr[:, -1] = 1
    return arr

def make_random_binary(size: int, rng: np.random.Generator) -> np.ndarray:
    """Random binary array using donut structure (0-centre, 1-border)."""
    return make_donut(size)

def make_random_pattern(pat_size: int, rng: np.random.Generator) -> np.ndarray:
    """Random binary pattern."""
    return rng.integers(0, 2, size=(pat_size, pat_size), dtype=np.int8)

def run_all(text: np.ndarray, pattern: np.ndarray):
    """Run every algorithm on (text, pattern). Returns dict of {name: (comp_count, result, wall_s)}."""
    results = {}
    for name, fn in ALGORITHMS.items():
        t0 = time.perf_counter()
        try:
            comp, pos = fn(text.copy(), pattern.copy())
        except Exception as e:
            comp, pos = -1, f'ERROR: {e}'
        wall = time.perf_counter() - t0
        results[name] = (comp, pos, wall)
    return results

def results_to_rows(results: dict, extra: dict) -> list:
    """Flatten run_all output into a list of dicts suitable for a DataFrame."""
    rows = []
    for algo, (comp, pos, wall) in results.items():
        row = {'algorithm': algo, 'comparisons': comp, 'match': pos, 'wall_sec': round(wall, 6)}
        row.update(extra)
        rows.append(row)
    return rows

print('Utilities defined.')

---
## Part 1 — Random Growth Ablation

In [ ]:
PATTERN_SIZES = [3, 5, 7, 13]
TEXT_SIZES    = [25, 50, 100, 200, 500, 1000]
SEEDS         = [42, 7, 123, 2024, 999]

random_rows = []

total = len(PATTERN_SIZES) * len(TEXT_SIZES) * len(SEEDS)
done  = 0

for pat_size in PATTERN_SIZES:
    for txt_size in TEXT_SIZES:
        if pat_size >= txt_size:
            done += len(SEEDS)
            continue                # skip invalid combos
        for seed in SEEDS:
            rng  = np.random.default_rng(seed)
            text    = make_donut(txt_size)
            pattern = make_random_pattern(pat_size, rng)

            res = run_all(text, pattern)
            extra = {'pattern_size': pat_size, 'text_size': txt_size, 'seed': seed}
            random_rows.extend(results_to_rows(res, extra))

            done += 1
            if done % 20 == 0:
                print(f'  {done}/{total} combos done...')

df_random = pd.DataFrame(random_rows)
df_random.to_csv('results_random.csv', index=False)
print(f'\nRandom ablation complete. Shape: {df_random.shape}')
df_random.head(10)

### 1a — Comparisons vs Text Size (per Pattern Size)

In [ ]:
algo_colors = {
    'Naive':         '#e74c3c',
    'Rabin-Karp':    '#3498db',
    'Vectorized RK': '#2ecc71',
    'Greedy KMP':    '#9b59b6',
    'Greedy BM':     '#f39c12',
    'Bird-KMP':      '#1abc9c',
    'Bird-BM':       '#e67e22',
}

# Average over seeds
agg = (df_random
       .groupby(['algorithm', 'pattern_size', 'text_size'])['comparisons']
       .mean()
       .reset_index())

fig, axes = plt.subplots(1, len(PATTERN_SIZES), figsize=(6 * len(PATTERN_SIZES), 5), sharey=False)
fig.suptitle('Comparisons vs Text Size — Random Donut (avg over seeds)', fontsize=15, fontweight='bold')

for ax, pat in zip(axes, PATTERN_SIZES):
    sub = agg[agg['pattern_size'] == pat]
    for algo in ALGORITHMS:
        d = sub[sub['algorithm'] == algo].sort_values('text_size')
        if d.empty:
            continue
        ax.plot(d['text_size'], d['comparisons'],
                marker='o', label=algo, color=algo_colors[algo], linewidth=2)
    ax.set_title(f'Pattern {pat}×{pat}', fontsize=12)
    ax.set_xlabel('Text Size (N×N)')
    ax.set_ylabel('Avg. Comparisons')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('random_comparisons_vs_textsize.png', dpi=150)
plt.show()
print('Saved: random_comparisons_vs_textsize.png')

### 1b — Comparisons vs Pattern Size (per Text Size)

In [ ]:
plot_text_sizes = [50, 100, 200, 500]

fig, axes = plt.subplots(1, len(plot_text_sizes),
                         figsize=(6 * len(plot_text_sizes), 5), sharey=False)
fig.suptitle('Comparisons vs Pattern Size — Random Donut (avg over seeds)', fontsize=15, fontweight='bold')

for ax, txt in zip(axes, plot_text_sizes):
    sub = agg[agg['text_size'] == txt]
    for algo in ALGORITHMS:
        d = sub[sub['algorithm'] == algo].sort_values('pattern_size')
        if d.empty:
            continue
        ax.plot(d['pattern_size'], d['comparisons'],
                marker='s', label=algo, color=algo_colors[algo], linewidth=2)
    ax.set_title(f'Text {txt}×{txt}', fontsize=12)
    ax.set_xlabel('Pattern Size (M×M)')
    ax.set_ylabel('Avg. Comparisons')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('random_comparisons_vs_patsize.png', dpi=150)
plt.show()
print('Saved: random_comparisons_vs_patsize.png')

### 1c — Wall-Clock Time Overview (Random)

In [ ]:
agg_time = (df_random
            .groupby(['algorithm', 'pattern_size', 'text_size'])['wall_sec']
            .mean()
            .reset_index())

fig, axes = plt.subplots(1, len(PATTERN_SIZES), figsize=(6 * len(PATTERN_SIZES), 5), sharey=False)
fig.suptitle('Wall-Clock Time vs Text Size — Random Donut (avg over seeds)', fontsize=15, fontweight='bold')

for ax, pat in zip(axes, PATTERN_SIZES):
    sub = agg_time[agg_time['pattern_size'] == pat]
    for algo in ALGORITHMS:
        d = sub[sub['algorithm'] == algo].sort_values('text_size')
        if d.empty:
            continue
        ax.plot(d['text_size'], d['wall_sec'],
                marker='o', label=algo, color=algo_colors[algo], linewidth=2)
    ax.set_title(f'Pattern {pat}×{pat}', fontsize=12)
    ax.set_xlabel('Text Size (N×N)')
    ax.set_ylabel('Avg. Wall-Clock (s)')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('random_wallclock.png', dpi=150)
plt.show()

---
## Part 2 — Real-World QR Codes
Search for a Tetrimino (L/J shape) in each QR code image:
```
 #
###
```

In [ ]:
# Tetrimino (L-shape): row 0 → [0,1,0]  but we use the given shape:
#   col: 0 1 2
# row 0:  0 1 0   <- the top prong
# row 1:  1 1 1   <- the full base
# Binary: 1 = dark module, 0 = light module

TETRIMINO = np.array([
    [0, 1, 0],
    [1, 1, 1],
], dtype=np.int8)

print('Tetrimino pattern:')
print(TETRIMINO)

QR_DIR = Path('qr')
qr_files = sorted(QR_DIR.glob('*.png'))[:100]   # at most 100
print(f'\nFound {len(qr_files)} QR PNG files in ./{QR_DIR}/')

In [ ]:
def png_to_binary(path: Path, threshold: int = 128) -> np.ndarray:
    """Load a PNG, convert to grayscale, threshold to binary int8 array.
    Dark pixels (< threshold) → 1, light → 0 (standard QR convention).
    """
    img = Image.open(path).convert('L')   # grayscale
    arr = np.array(img, dtype=np.int8)
    binary = (arr < threshold).astype(np.int8)  # 1 = dark
    return binary

# Quick sanity check on first QR
if qr_files:
    sample = png_to_binary(qr_files[0])
    print(f'Sample QR shape: {sample.shape}, unique values: {np.unique(sample)}')
else:
    print('WARNING: No QR files found — skipping QR section.')

In [ ]:
qr_rows = []

for idx, qr_path in enumerate(qr_files):
    try:
        text_qr = png_to_binary(qr_path)
    except Exception as e:
        print(f'  Could not load {qr_path.name}: {e}')
        continue

    res = run_all(text_qr, TETRIMINO)
    extra = {'qr_file': qr_path.name,
             'qr_idx':  idx,
             'text_h':  text_qr.shape[0],
             'text_w':  text_qr.shape[1]}
    qr_rows.extend(results_to_rows(res, extra))

    if (idx + 1) % 10 == 0:
        print(f'  {idx+1}/{len(qr_files)} QR codes processed...')

df_qr = pd.DataFrame(qr_rows)
df_qr.to_csv('results_qr.csv', index=False)
print(f'\nQR section complete. Shape: {df_qr.shape}')
df_qr.head(10)

### 2a — Average Comparisons per Algorithm Across All QR Codes

In [ ]:
if not df_qr.empty:
    agg_qr = (df_qr.groupby('algorithm')['comparisons']
              .mean()
              .reindex(list(ALGORITHMS.keys()))
              .reset_index())
    agg_qr.columns = ['algorithm', 'avg_comparisons']

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = [algo_colors[a] for a in agg_qr['algorithm']]
    bars = ax.bar(agg_qr['algorithm'], agg_qr['avg_comparisons'], color=colors, edgecolor='black')
    ax.set_title('Avg. Comparisons on QR Codes — Tetrimino Search', fontsize=14, fontweight='bold')
    ax.set_xlabel('Algorithm')
    ax.set_ylabel('Avg. Comparisons')
    ax.bar_label(bars, fmt='%.0f', padding=3, fontsize=9)
    plt.xticks(rotation=20, ha='right')
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig('qr_avg_comparisons.png', dpi=150)
    plt.show()
    print('Saved: qr_avg_comparisons.png')
else:
    print('No QR data to plot.')

### 2b — Per-QR-Code Comparisons (all algorithms)

In [ ]:
if not df_qr.empty:
    fig, ax = plt.subplots(figsize=(max(12, len(qr_files) // 3), 6))

    for algo in ALGORITHMS:
        sub = df_qr[df_qr['algorithm'] == algo].sort_values('qr_idx')
        ax.plot(sub['qr_idx'], sub['comparisons'],
                label=algo, color=algo_colors[algo], linewidth=1.5, marker='.', markersize=4)

    ax.set_title('Comparisons per QR Code — Tetrimino Search', fontsize=14, fontweight='bold')
    ax.set_xlabel('QR Code Index')
    ax.set_ylabel('Comparisons')
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig('qr_per_image_comparisons.png', dpi=150)
    plt.show()
else:
    print('No QR data to plot.')

### 2c — Wall-Clock Time per Algorithm (QR Codes)

In [ ]:
if not df_qr.empty:
    agg_qr_time = (df_qr.groupby('algorithm')['wall_sec']
                   .mean()
                   .reindex(list(ALGORITHMS.keys()))
                   .reset_index())
    agg_qr_time.columns = ['algorithm', 'avg_wall_sec']

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = [algo_colors[a] for a in agg_qr_time['algorithm']]
    bars = ax.bar(agg_qr_time['algorithm'], agg_qr_time['avg_wall_sec'], color=colors, edgecolor='black')
    ax.set_title('Avg. Wall-Clock Time on QR Codes', fontsize=14, fontweight='bold')
    ax.set_xlabel('Algorithm')
    ax.set_ylabel('Avg. Wall-Clock (s)')
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
    plt.xticks(rotation=20, ha='right')
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig('qr_wallclock.png', dpi=150)
    plt.show()
else:
    print('No QR data to plot.')

---
## Part 3 — Contrived Cases
All-1s text with a 3×3 block of 0s placed in the **bottom-right corner** (last 3 rows & last 3 columns).
Pattern: 3×3 block of all 0s.
Text sizes mirror the random ablation.

In [ ]:
CONTRIVED_PATTERN = np.zeros((3, 3), dtype=np.int8)

print('Contrived pattern (3×3 zeros):')
print(CONTRIVED_PATTERN)

In [ ]:
def make_contrived_text(size: int) -> np.ndarray:
    """All-1s array with a 3×3 block of 0s in the bottom-right corner."""
    arr = np.ones((size, size), dtype=np.int8)
    arr[-3:, -3:] = 0
    return arr

contrived_rows = []

for txt_size in TEXT_SIZES:
    if txt_size < 3:
        continue
    text_c = make_contrived_text(txt_size)
    res = run_all(text_c, CONTRIVED_PATTERN)
    extra = {'text_size': txt_size, 'pattern_size': 3, 'case': 'contrived'}
    contrived_rows.extend(results_to_rows(res, extra))
    print(f'  text_size={txt_size} done. expected match at ({txt_size-3}, {txt_size-3}).')

df_contrived = pd.DataFrame(contrived_rows)
df_contrived.to_csv('results_contrived.csv', index=False)
print(f'\nContrived section complete. Shape: {df_contrived.shape}')
df_contrived

### 3a — Comparisons vs Text Size (Contrived Worst-Case)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for algo in ALGORITHMS:
    sub = df_contrived[df_contrived['algorithm'] == algo].sort_values('text_size')
    ax.plot(sub['text_size'], sub['comparisons'],
            marker='o', label=algo, color=algo_colors[algo], linewidth=2)

ax.set_title('Comparisons vs Text Size — Contrived (all-1s, 3×3 zero block at end)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Text Size (N×N)')
ax.set_ylabel('Comparisons')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('contrived_comparisons_vs_textsize.png', dpi=150)
plt.show()
print('Saved: contrived_comparisons_vs_textsize.png')

### 3b — Wall-Clock Time (Contrived)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for algo in ALGORITHMS:
    sub = df_contrived[df_contrived['algorithm'] == algo].sort_values('text_size')
    ax.plot(sub['text_size'], sub['wall_sec'],
            marker='s', label=algo, color=algo_colors[algo], linewidth=2)

ax.set_title('Wall-Clock Time vs Text Size — Contrived', fontsize=13, fontweight='bold')
ax.set_xlabel('Text Size (N×N)')
ax.set_ylabel('Wall-Clock (s)')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('contrived_wallclock.png', dpi=150)
plt.show()

---
## Summary Table — All Sections

In [ ]:
# ── Random: overall average per algorithm
s_random = (df_random
            .groupby('algorithm')['comparisons']
            .mean()
            .rename('random_avg_comparisons'))

# ── QR: overall average per algorithm
if not df_qr.empty:
    s_qr = (df_qr
            .groupby('algorithm')['comparisons']
            .mean()
            .rename('qr_avg_comparisons'))
else:
    s_qr = pd.Series(dtype=float, name='qr_avg_comparisons')

# ── Contrived: overall average per algorithm
s_contrived = (df_contrived
               .groupby('algorithm')['comparisons']
               .mean()
               .rename('contrived_avg_comparisons'))

summary = pd.concat([s_random, s_qr, s_contrived], axis=1).round(1)
summary.index.name = 'algorithm'
summary = summary.reindex(list(ALGORITHMS.keys()))
print('=== Summary: Average Comparisons per Section ===')
print(summary.to_string())
summary.to_csv('results_summary.csv')
print('\nSaved: results_summary.csv')

In [ ]:
# Side-by-side bar chart comparing all three sections
cols_present = [c for c in ['random_avg_comparisons', 'qr_avg_comparisons', 'contrived_avg_comparisons']
                if c in summary.columns and summary[c].notna().any()]

x = np.arange(len(summary))
width = 0.25
section_colors = ['#3498db', '#e74c3c', '#2ecc71']
section_labels = ['Random (avg)', 'QR Codes (avg)', 'Contrived (avg)']

fig, ax = plt.subplots(figsize=(13, 6))
for k, (col, color, label) in enumerate(zip(cols_present, section_colors, section_labels)):
    vals = summary[col].fillna(0).values
    offset = (k - len(cols_present) / 2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=label, color=color, alpha=0.85, edgecolor='black')

ax.set_title('Average Comparisons — All Sections & Algorithms', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(summary.index, rotation=20, ha='right')
ax.set_ylabel('Avg. Comparisons')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('summary_all_sections.png', dpi=150)
plt.show()
print('Saved: summary_all_sections.png')